# IPD Dataset Pipeline — Interactive Notebook

**Phase 0 / Phase 1: Merge · Audit · Deduplicate · Split**

---

## Instructions
Run **one cell at a time**, in order. Read the output of each stage before proceeding.  
Stages with `⚠️ DECISION POINT` require you to inspect the output and decide before continuing.

| Stage | Description |
|:---:|:---|
| 0 | Configuration and paths |
| 1 | Pre-audit manifest — full inventory |
| 2 | File integrity check |
| 3 | Exact duplicate removal (SHA-256) |
| 4a | Perceptual hash computation |
| 4b | Near-duplicate family detection |
| 5 | Rice Healthy integration |
| 6 | Group-stratified train/val/test split |
| 7 | **Leakage audit (MUST PASS before continuing)** |
| 8 | Build final directory structure |
| 9 | Save locked split manifest |
| 10 | Final statistics report + verification |

---
## Stage 0 · Configuration

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import pandas as pd

# ── Project root: automatically locate folder containing finaldataset ──────
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "finaldataset").is_dir() and (candidate / "Rice___Healthy").is_dir():
        PROJECT_ROOT = candidate.resolve()
        break
else:
    PROJECT_ROOT = Path("..").resolve()

print(f"Project root : {PROJECT_ROOT}")

# Add src/ to path so dataset_pipeline.py can be imported
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Import pipeline module
import dataset_pipeline as dp
print(f"dataset_pipeline module loaded from: {dp.__file__}")

# ── Source directories ──────────────────────────────────────────────────────
FINALDATASET_ROOT   = PROJECT_ROOT / "finaldataset"
RICE_HEALTHY_SOURCE = PROJECT_ROOT / "Rice___Healthy"

# Rice disease source directories (train + val + test merged for re-splitting)
RICE_DISEASE_DIRS = [
    FINALDATASET_ROOT / "rice_dataset" / "train",
    FINALDATASET_ROOT / "rice_dataset" / "val",
    FINALDATASET_ROOT / "rice_dataset" / "test",
]

# Potato and tomato source directories (flat class folders)
POTATO_SOURCE = FINALDATASET_ROOT / "potato_dataset"
TOMATO_SOURCE = FINALDATASET_ROOT / "tomato_dataset"

# ── Output directories ──────────────────────────────────────────────────────
# Final re-split dataset will be written here.
# Using a new 'clean_dataset' directory so originals are preserved
# until the full pipeline has been verified.
CLEAN_OUTPUT_ROOT = PROJECT_ROOT / "clean_dataset"

MANIFESTS_DIR  = FINALDATASET_ROOT / "manifests"
STAGING_DIR    = FINALDATASET_ROOT / "rice_dataset" / "_staging" / "healthy"

# Manifest file paths
PRE_AUDIT_CSV         = MANIFESTS_DIR / "pre_audit_manifest.csv"
CORRUPT_CSV           = MANIFESTS_DIR / "corrupt_files.csv"
EXACT_DUP_CSV         = MANIFESTS_DIR / "exact_duplicates.csv"
PHASH_FAMILIES_CSV    = MANIFESTS_DIR / "phash_duplicate_families.csv"
SPLIT_MANIFEST_CSV    = MANIFESTS_DIR / "split_manifest_v1.csv"

# ── Verify all source directories exist ────────────────────────────────────
print("\nChecking source directories...")
dirs_to_check = [
    RICE_HEALTHY_SOURCE,
    POTATO_SOURCE,
    TOMATO_SOURCE,
] + RICE_DISEASE_DIRS

all_ok = True
for d in dirs_to_check:
    exists = d.exists()
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {d}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ All source directories found. Ready to proceed.")
else:
    print("\n❌ Some source directories are missing. Fix paths before continuing.")

Project root : C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd
dataset_pipeline module loaded from: C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\src\dataset_pipeline.py

Checking source directories...
  ✓  C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\Rice___Healthy
  ✓  C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\potato_dataset
  ✓  C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\tomato_dataset
  ✓  C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\rice_dataset\train
  ✓  C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\rice_dataset\val
  ✓  C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\rice_dataset\test

✅ All source directories found. Ready to proceed.


In [2]:
# Verify dataset_pipeline and pandas
import dataset_pipeline as dp
import pandas as pd
print(f"✓ dataset_pipeline ready from: {dp.__file__}")
print(f"✓ pandas version: {pd.__version__}")

✓ dataset_pipeline ready from: C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\src\dataset_pipeline.py
✓ pandas version: 3.0.5


---
## Stage 1 · Pre-Audit Manifest

Walk all source directories and build a full inventory row by row.  
No files are modified. This is the baseline record.

In [3]:
# Build the source_dirs dict.
# Each entry: "path_string" → {"crop": ..., "class_label": ..., (optional) "source_override": ...}

source_dirs = {}

# ── Rice disease classes (pulling from all 3 existing splits) ───────────────
RICE_CLASSES = ["blast", "blight", "brown_spot"]
for split in ["train", "val", "test"]:
    for cls in RICE_CLASSES:
        d = FINALDATASET_ROOT / "rice_dataset" / split / cls
        if d.exists():
            source_dirs[str(d)] = {
                "crop"        : "rice",
                "class_label" : cls,
                "source_override": dp.SOURCE_RICE_DISEASE,
            }

# ── Rice healthy (external field photos) ────────────────────────────────────
if RICE_HEALTHY_SOURCE.exists():
    source_dirs[str(RICE_HEALTHY_SOURCE)] = {
        "crop"           : "rice",
        "class_label"    : "healthy",
        "source_override": dp.SOURCE_RICE_HEALTHY_FIELD,
    }

# ── Potato classes ───────────────────────────────────────────────────────────
POTATO_CLASSES = {
    "Potato___early_blight" : "early_blight",
    "Potato___healthy"      : "healthy",
    "Potato___late_blight"  : "late_blight",
}
for folder_name, class_label in POTATO_CLASSES.items():
    d = POTATO_SOURCE / folder_name
    if d.exists():
        source_dirs[str(d)] = {
            "crop"        : "potato",
            "class_label" : class_label,
        }

# ── Tomato classes ───────────────────────────────────────────────────────────
TOMATO_CLASSES = {
    "Tomato___early_blight" : "early_blight",
    "Tomato___healthy"      : "healthy",
    "Tomato___late_blight"  : "late_blight",
}
for folder_name, class_label in TOMATO_CLASSES.items():
    d = TOMATO_SOURCE / folder_name
    if d.exists():
        source_dirs[str(d)] = {
            "crop"        : "tomato",
            "class_label" : class_label,
        }

print(f"Total source directories configured: {len(source_dirs)}")
for k in source_dirs:
    m = source_dirs[k]
    print(f"  {Path(k).name:<35}  crop={m['crop']:<8}  class={m['class_label']}")

Total source directories configured: 16
  blast                                crop=rice      class=blast
  blight                               crop=rice      class=blight
  brown_spot                           crop=rice      class=brown_spot
  blast                                crop=rice      class=blast
  blight                               crop=rice      class=blight
  brown_spot                           crop=rice      class=brown_spot
  blast                                crop=rice      class=blast
  blight                               crop=rice      class=blight
  brown_spot                           crop=rice      class=brown_spot
  Rice___Healthy                       crop=rice      class=healthy
  Potato___early_blight                crop=potato    class=early_blight
  Potato___healthy                     crop=potato    class=healthy
  Potato___late_blight                 crop=potato    class=late_blight
  Tomato___early_blight                crop=tomato    class=early_b

In [4]:
# Run Stage 0/1: build manifest
raw_manifest = dp.build_pre_audit_manifest(
    source_dirs   = source_dirs,
    output_csv    = PRE_AUDIT_CSV,
)
print(f"\nManifest shape: {raw_manifest.shape}")
raw_manifest.head(3)

20:31:38  INFO      ============================================================
20:31:38  INFO      STAGE 0: Building pre-audit manifest
20:31:38  INFO      ============================================================
20:31:38  INFO        Scanning C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\rice_dataset\train\blast → 672 images [crop=rice, class=blast]
20:31:38  INFO        Scanning C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\rice_dataset\train\blight → 898 images [crop=rice, class=blight]
20:31:38  INFO        Scanning C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\rice_dataset\train\brown_spot → 840 images [crop=rice, class=brown_spot]
20:31:38  INFO        Scanning C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\rice_dataset\val\blast → 144 images [crop=rice, class=blast]
20:31:38  INFO        Scanning C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\i


Manifest shape: (16390, 14)


,image_id,original_path,crop,class_label,source,sha256,file_size_bytes,width,height,format,integrity_ok,phash,group_id,partition
0,blast_train_00001,C:\Users\Dhruv Dube\Desktop\New folder\IPD rea...,rice,blast,RiceDisease_Unknown,68dac086a6ad4579ae08c013238333222943194af085b9...,15475,None,None,jpg,None,None,None,None
1,blast_train_00002,C:\Users\Dhruv Dube\Desktop\New folder\IPD rea...,rice,blast,RiceDisease_Unknown,cca252a8321516dfa839ca8df00ab235be86a5f2888076...,19645,None,None,jpg,None,None,None,None
2,blast_train_00003,C:\Users\Dhruv Dube\Desktop\New folder\IPD rea...,rice,blast,RiceDisease_Unknown,979d051602f7a039161ed0a6992c7365a83376f87c28d9...,15922,None,None,jpg,None,None,None,None


---
## Stage 2 · File Integrity Check

Every image is fully decoded with PIL.  
Corrupt files are quarantined and removed from the working manifest.  

⚠️ **DECISION POINT**: If `corrupt_files.csv` contains images, inspect them  
before proceeding. Do not delete them from disk — just note they are excluded.

In [5]:
clean_manifest, corrupt_df = dp.check_file_integrity(
    manifest            = raw_manifest,
    output_corrupt_csv  = CORRUPT_CSV,
)

print(f"\nClean images : {len(clean_manifest):,}")
print(f"Corrupt files: {len(corrupt_df):,}")

if not corrupt_df.empty:
    print("\n⚠️  Corrupt files found — review before proceeding:")
    display(corrupt_df[["original_path", "crop", "class_label"]])

20:31:44  INFO      ============================================================
20:31:44  INFO      STAGE 1: File integrity check
20:31:44  INFO      ============================================================
  Integrity check: 100%|██████████| 16390/16390 [00:10<00:00, 1567.17it/s]
20:31:54  INFO      
  ✓ Clean files   : 16390
20:31:54  INFO        ✗ Corrupt files : 0



Clean images : 16,390
Corrupt files: 0


---
## Stage 3 · Exact Duplicate Removal (SHA-256)

Byte-identical images are collapsed to one canonical copy.  

⚠️ **DECISION POINT**: If any SHA-256 hash appears in more than one class label,  
a **CRITICAL ERROR** will be logged. This is a label consistency problem that  
requires manual review before proceeding.

In [6]:
dedup_manifest, exact_removed_df = dp.remove_exact_duplicates(
    manifest              = clean_manifest,
    output_duplicates_csv = EXACT_DUP_CSV,
)

print(f"\nImages after exact dedup: {len(dedup_manifest):,}")
print(f"Exact duplicates removed: {len(exact_removed_df):,}")

# Show breakdown by crop and class
print("\nImages remaining by crop/class:")
display(
    dedup_manifest.groupby(["crop", "class_label"]).size()
    .reset_index(name="count")
    .sort_values(["crop", "class_label"])
)

20:31:57  INFO      ============================================================
20:31:57  INFO      STAGE 2: Exact duplicate removal (SHA-256)
20:31:57  INFO      ============================================================
20:31:57  INFO      
  Total images (pre-dedup) : 16390
20:31:57  INFO        Exact duplicate rows removed: 0
20:31:57  INFO        Clean images remaining   : 16390
20:31:57  INFO        Duplicate list saved → C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\manifests\exact_duplicates.csv



Images after exact dedup: 16,390
Exact duplicates removed: 0

Images remaining by crop/class:


,crop,class_label,count
0,potato,early_blight,2627
1,potato,healthy,2263
2,potato,late_blight,2082
3,rice,blast,960
4,rice,blight,1284
5,rice,brown_spot,1200
6,rice,healthy,1488
7,tomato,early_blight,1000
8,tomato,healthy,1585
9,tomato,late_blight,1901


---
## Stage 4a · Perceptual Hash Computation

Compute a 64-bit perceptual hash (pHash) for every surviving image.  
This is needed for near-duplicate detection in the next stage.

> This stage takes the longest — expect several minutes to hours depending  
> on the total image count. The progress bar shows ETA.

In [7]:
phash_manifest = dp.compute_phashes(
    manifest  = dedup_manifest,
    hash_size = 8,    # 64-bit hash — do not change without re-reading Agent.md
)

null_phash = phash_manifest["phash"].isna().sum()
print(f"\npHash computed: {len(phash_manifest) - null_phash:,} / {len(phash_manifest):,}")
if null_phash > 0:
    print(f"⚠️  {null_phash} images could not be hashed — they will use SHA-256 as group_id.")

20:32:02  INFO      ============================================================
20:32:02  INFO      STAGE 3a: Computing perceptual hashes (pHash)
20:32:02  INFO      ============================================================
20:32:02  INFO        Computing pHash for 16390 images...
  pHash: 100%|██████████| 16390/16390 [00:12<00:00, 1300.61it/s]
20:32:15  INFO      
  ✓ pHash available for 16390 / 16390 images



pHash computed: 16,390 / 16,390


---
## Stage 4b · Near-Duplicate Family Detection

Build a graph where images with pHash Hamming distance ≤ 8 are connected.  
Each connected component = a **family** that must stay in one partition.

⚠️ **DECISION POINT**: Check the family size distribution for `Rice___Healthy`.  
If many families have sizes > 5, it means the field session produced many  
near-identical shots. The number of unique groups determines effective healthy  
image count for splitting.

In [8]:
grouped_manifest = dp.find_near_duplicate_families(
    manifest   = phash_manifest,
    threshold  = 8,
    output_csv = PHASH_FAMILIES_CSV,
)

print("\nGroup summary:")
print(grouped_manifest.groupby(["crop", "class_label"])[["group_id"]].nunique().rename(columns={"group_id": "unique_groups"}))

print("\n\nRice Healthy — family size distribution:")
rice_healthy = grouped_manifest[
    (grouped_manifest["crop"] == "rice") &
    (grouped_manifest["class_label"] == "healthy")
]
print(f"  Total images  : {len(rice_healthy):,}")
print(f"  Unique groups : {rice_healthy['group_id'].nunique():,}")
print(f"  Family sizes  :")
print(rice_healthy["family_size"].value_counts().sort_index().head(20))

20:32:17  INFO      ============================================================
20:32:17  INFO      STAGE 3b: Near-duplicate family detection (Hamming <= 8, scoped per class)
20:32:17  INFO      ============================================================
20:32:23  INFO         potato / early_blight     images: 2627  pairs: 3,449,251  near-dups:  1464  unique groups: 1826
20:32:27  INFO         potato / healthy          images: 2263  pairs: 2,559,453  near-dups:   269  unique groups: 2020
20:32:31  INFO         potato / late_blight      images: 2082  pairs: 2,166,321  near-dups:   386  unique groups: 1741
20:32:32  INFO           rice / blast            images:  960  pairs: 460,320  near-dups:   504  unique groups:  474
20:32:34  INFO           rice / blight           images: 1284  pairs: 823,686  near-dups:  1236  unique groups:  514
20:32:35  INFO           rice / brown_spot       images: 1200  pairs: 719,400  near-dups:   602  unique groups:  604
20:32:37  INFO           rice / hea


Group summary:
                     unique_groups
crop   class_label                
potato early_blight           1826
       healthy                2020
       late_blight            1741
rice   blast                   474
       blight                  514
       brown_spot              604
       healthy                 623
tomato early_blight            994
       healthy                1526
       late_blight            1841


Rice Healthy — family size distribution:
  Total images  : 1,488
  Unique groups : 623
  Family sizes  :
family_size
1      564
2       64
3       18
4       12
5       10
6       12
7       21
11      11
12      12
15      15
18      36
23      23
28      28
40      40
97      97
254    254
271    271
Name: count, dtype: int64


---
## Stage 5 · Rice Healthy Integration

Confirm rice healthy images are in the manifest and stage them for inspection.  
No files are moved to their final location yet.

In [9]:
integrated_manifest = dp.integrate_rice_healthy(
    manifest                = grouped_manifest,
    rice_healthy_source_dir = RICE_HEALTHY_SOURCE,
    staging_dir             = STAGING_DIR,
)

print("\nRice class totals after integration:")
print(
    integrated_manifest[integrated_manifest["crop"] == "rice"]
    .groupby("class_label").size()
    .rename("images")
    .to_string()
)

20:32:51  INFO      ============================================================
20:32:51  INFO      STAGE 4: Rice Healthy Integration
20:32:51  INFO      ============================================================
20:32:51  INFO        Rice healthy images in manifest: 1488
20:32:51  INFO        Source: C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\Rice___Healthy
20:32:51  INFO      
  Healthy images by source:
source
RiceHealthyField_20190419    1488
20:32:51  INFO      
  Family size distribution for rice healthy:
family_size
1      564
2       64
3       18
4       12
5       10
6       12
7       21
11      11
12      12
15      15
18      36
23      23
28      28
40      40
97      97
254    254
271    271
20:32:51  INFO      
  Unique group_ids (will appear once in the split): 623
20:32:51  INFO        ⚠️  Image count (1488) vs unique groups (623) — the split operates on groups, so effectively 623 'units' will be split 70/15/15.
20:32:51  INFO      
  Staging di


Rice class totals after integration:
class_label
blast          960
blight        1284
brown_spot    1200
healthy       1488


---
## Stage 6 · Group-Stratified Train / Val / Test Split

Split at the **group level** — all images in the same group stay in the same  
partition. Seed is fixed at 42 for reproducibility.

Review the printed split table carefully before proceeding.

In [10]:
split_manifest = dp.stratified_group_split(
    manifest     = integrated_manifest,
    train_ratio  = 0.70,
    val_ratio    = 0.15,
    test_ratio   = 0.15,
    seed         = 42,
)

print("\nPartition counts:")
print(split_manifest["partition"].value_counts())

print("\nFull split breakdown by crop/class/partition:")
pivot_check = (
    split_manifest
    .groupby(["crop", "class_label", "partition"])
    .size()
    .unstack(fill_value=0)
)
for col in ["train", "val", "test"]:
    if col not in pivot_check.columns:
        pivot_check[col] = 0
pivot_check = pivot_check[["train", "val", "test"]]
pivot_check["TOTAL"] = pivot_check.sum(axis=1)
display(pivot_check)

20:32:58  INFO      ============================================================
20:32:58  INFO      STAGE 5: Group-stratified train/val/test split
20:32:58  INFO        Ratios → train=70%  val=15%  test=15%
20:32:58  INFO        Random seed: 42
20:32:58  INFO      ============================================================
20:32:58  INFO         potato / early_blight     groups: 1826  train:  2054  val:  282  test:  291
20:32:58  INFO         potato / healthy          groups: 2020  train:  1605  val:  321  test:  337
20:32:58  INFO         potato / late_blight      groups: 1741  train:  1453  val:  346  test:  283
20:32:58  INFO           rice / blast            groups:  474  train:   674  val:  142  test:  144
20:32:58  INFO           rice / blight           groups:  514  train:   891  val:  196  test:  197
20:32:58  INFO           rice / brown_spot       groups:  604  train:   846  val:  176  test:  178
20:32:58  INFO           rice / healthy          groups:  623  train:   904  va


Partition counts:
partition
train    11575
test      2561
val       2254
Name: count, dtype: int64

Full split breakdown by crop/class/partition:


partition            train  val  test  TOTAL
crop   class_label                          
potato early_blight   2054  282   291   2627
       healthy        1605  321   337   2263
       late_blight    1453  346   283   2082
rice   blast           674  142   144    960
       blight          891  196   197   1284
       brown_spot      846  176   178   1200
       healthy         904  122   462   1488
tomato early_blight    702  149   149   1000
       healthy        1117  232   236   1585
       late_blight    1329  288   284   1901

---
## Stage 7 · Leakage Audit ⚠️ CRITICAL GATE

**DO NOT proceed to Stage 8 if this audit fails.**

This audit checks:
1. Zero SHA-256 overlap across partitions
2. Zero group_id shared across partition boundaries  
3. Zero pHash near-duplicates between train ↔ test  
4. All classes present in all partitions  
5. All rows have a valid partition assigned  

Only if `audit_passed == True` should you run the cells below.

In [11]:
# Note: Check 3 compares near-duplicate leakage between train and test per class.
# With class-scoped grouping, this check runs cleanly in ~1-2 seconds.
audit_passed = dp.run_leakage_audit(split_manifest)

print("\n" + ("=" * 60))
if audit_passed:
    print("✅  AUDIT PASSED — you may proceed to Stage 8.")
else:
    print("❌  AUDIT FAILED — STOP. Do not run Stage 8.")
    print("    Review the error messages above and fix before continuing.")
print("=" * 60)

20:33:08  INFO      ============================================================
20:33:08  INFO      STAGE 6: Cross-partition leakage audit
20:33:08  INFO      ============================================================
20:33:08  INFO        [PASS] Check 0: All rows have a partition assigned.
20:33:08  INFO        [PASS] Check 1: No SHA-256 overlap between train ↔ val.
20:33:08  INFO        [PASS] Check 1: No SHA-256 overlap between train ↔ test.
20:33:08  INFO        [PASS] Check 1: No SHA-256 overlap between val ↔ test.
20:33:08  INFO        [PASS] Check 2: Zero group_id overlap across partition boundaries.
20:33:08  INFO        Check 3: pHash Hamming ≤ 8 between train ↔ test (per class)...
20:33:14  INFO        [PASS] Check 3: No pHash leakage between train ↔ test in any class.
20:33:14  INFO        Check 4: All classes present in all partitions...
20:33:14  INFO        [PASS] Check 4: All classes present in all partitions.
20:33:14  INFO      
20:33:14  INFO        ✅  AUDIT PASSED


✅  AUDIT PASSED — you may proceed to Stage 8.


---
## Stage 8 · Build Final Directory Structure

**Only run this cell if `audit_passed == True`.**

Files are COPIED (not moved) to `clean_dataset/`.  
The original `finaldataset/` directories are left unchanged.

Run with `dry_run=True` first to preview the operations without touching files.

In [12]:
if not audit_passed:
    raise RuntimeError(
        "Leakage audit did not pass. "
        "Fix all reported issues before running this cell."
    )

# ── DRY RUN first — preview without writing any files ──────────────────────
# Comment this out and uncomment the block below to do the real run.

# dp.build_final_directories(
#     split_manifest = split_manifest,
#     output_root    = CLEAN_OUTPUT_ROOT,
#     dry_run        = True,
# )

# ── REAL RUN — copies files to clean_dataset/ ──────────────────────────────
# Uncomment only after reviewing dry run output.

dp.build_final_directories(
    split_manifest = split_manifest,
    output_root    = CLEAN_OUTPUT_ROOT,
    dry_run        = False,    # Set True first to preview
)

print("\nFile copy complete.")

20:33:20  INFO      ============================================================
20:33:20  INFO      STAGE 7: Build final directories
20:33:20  INFO      ============================================================
  Copying: 100%|██████████| 16390/16390 [00:13<00:00, 1207.00it/s]
20:33:33  INFO      
  ✓ Files copied  : 16390
20:33:33  INFO        ⊘ Files skipped : 0 (already exist)
20:33:33  INFO        ✓ Zero copy errors.



File copy complete.


---
## Stage 9 · Save Locked Split Manifest

This manifest is the immutable record.  
**Once saved, it must not be regenerated or overwritten.**  
If it already exists, the function will raise `FileExistsError` to protect it.

In [13]:
if not audit_passed:
    raise RuntimeError("Leakage audit did not pass. Cannot save manifest.")

dp.save_split_manifest(
    split_manifest = split_manifest,
    output_path    = SPLIT_MANIFEST_CSV,
    overwrite      = False,    # ← NEVER change to True unless you have explicit approval
)

print(f"\nSplit manifest locked at: {SPLIT_MANIFEST_CSV}")
print(f"Rows: {len(split_manifest):,}")

20:33:38  INFO        ✓ Split manifest LOCKED → C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\manifests\split_manifest_v1.csv
20:33:38  INFO          Rows: 16390
20:33:38  INFO          Columns: ['image_id', 'original_path', 'crop', 'class_label', 'source', 'sha256', 'file_size_bytes', 'width', 'height', 'format', 'integrity_ok', 'phash', 'group_id', 'partition', 'family_size']



Split manifest locked at: C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\manifests\split_manifest_v1.csv
Rows: 16,390


---
## Stage 10 · Final Statistics Report + Verification

In [14]:
# Generate and print the full statistics report
report = dp.generate_statistics_report(split_manifest)

# Save the report to a text file for reference (UTF-8 encoding specified for Windows compatibility)
report_path = MANIFESTS_DIR / "split_statistics_report.txt"
report_path.write_text(report, encoding="utf-8")
print(f"\nReport saved → {report_path}")

20:33:43  INFO      ======================================================================
IPD DATASET FINAL STATISTICS REPORT

──────────────────────────────────────────────────────────────────────
  CROP: POTATO
──────────────────────────────────────────────────────────────────────

partition     train  val  test  TOTAL
class_label                          
early_blight   2054  282   291   2627
healthy        1605  321   337   2263
late_blight    1453  346   283   2082

  Source breakdown:
partition     train  val  test  TOTAL
source                               
PlantVillage   5112  949   911   6972

──────────────────────────────────────────────────────────────────────
  CROP: RICE
──────────────────────────────────────────────────────────────────────

partition    train  val  test  TOTAL
class_label                         
blast          674  142   144    960
blight         891  196   197   1284
brown_spot     846  176   178   1200
healthy        904  122   462   1488

  Source 


Report saved → C:\Users\Dhruv Dube\Desktop\New folder\IPD reaseach papers\ipd\finaldataset\manifests\split_statistics_report.txt


In [15]:
# Verify the output directory structure by counting files per class per partition
dp.verify_output_tree(CLEAN_OUTPUT_ROOT)

20:34:02  INFO      ============================================================
20:34:02  INFO      OUTPUT DIRECTORY VERIFICATION
20:34:02  INFO      ============================================================
20:34:02  INFO      
  potato_dataset/
20:34:02  INFO          test/
20:34:02  INFO            early_blight                      291 images
20:34:02  INFO            healthy                           337 images
20:34:02  INFO            late_blight                       283 images
20:34:02  INFO          train/
20:34:02  INFO            early_blight                    2,054 images
20:34:02  INFO            healthy                         1,605 images
20:34:02  INFO            late_blight                     1,453 images
20:34:02  INFO          val/
20:34:02  INFO            early_blight                      282 images
20:34:02  INFO            healthy                           321 images
20:34:02  INFO            late_blight                       346 images
20:34:02  INFO      

In [16]:
# ── Final cross-check: file count in output vs manifest count ──────────────
from pathlib import Path

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

actual_count = sum(
    1 for f in CLEAN_OUTPUT_ROOT.rglob("*")
    if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
)

manifest_count = split_manifest["partition"].notna().sum()

print(f"Files on disk  : {actual_count:,}")
print(f"Manifest rows  : {manifest_count:,}")

if actual_count == manifest_count:
    print("\n✅ File count matches manifest. Pipeline complete.")
else:
    print(f"\n⚠️  Mismatch: {abs(actual_count - manifest_count)} files differ from manifest count.")
    print("   Review copy errors in Stage 8 output.")

Files on disk  : 16,390
Manifest rows  : 16,390

✅ File count matches manifest. Pipeline complete.


---
## Pipeline Complete ✅

After this notebook runs successfully, the following artifacts exist:

```
clean_dataset/
├── potato_dataset/
│   ├── train/ { early_blight / healthy / late_blight }
│   ├── val/   { early_blight / healthy / late_blight }
│   └── test/  { early_blight / healthy / late_blight }
├── tomato_dataset/
│   └── ...
└── rice_dataset/
    ├── train/ { blast / blight / brown_spot / healthy }  ← healthy NOW INCLUDED
    ├── val/   { blast / blight / brown_spot / healthy }
    └── test/  { blast / blight / brown_spot / healthy }

finaldataset/manifests/
├── pre_audit_manifest.csv       ← baseline inventory
├── corrupt_files.csv            ← files that failed PIL decode
├── exact_duplicates.csv         ← SHA-256 duplicate families
├── phash_duplicate_families.csv ← near-duplicate groups
├── split_manifest_v1.csv        ← LOCKED (one row per image, with partition)
└── split_statistics_report.txt  ← human-readable summary
```

**Next step**: Phase 3 — EfficientNetB3 teacher training  
Point your training script at `clean_dataset/<crop>_dataset/` using `train/`, `val/`, `test/`.